# Probing Inside-Outside Structure in a Transformer Trained on the RHM

For a grammar of depth $L$ and a $2L$-layer transformer, the hypothesis is:
- **Layers 1…L (inside pass):** build level-$\ell$ representations bottom-up; representations remain ambiguous about the root.
- **Layers L+1…2L (outside pass):** propagate global context downward, eliminating spurious latents.

We test this by training linear probes at each layer to predict the ground-truth latents at every grammar level ($\ell = 1, 2, 3 = L$) from the residual stream.

**Probe dataset:** fully-ternary sequences generated on-the-fly from the grammar, so every token position has a well-defined parse-tree ancestor at every level.

**Expected signature of inside-outside:**
| Layer range | Root probe ($\ell=L$) | Level-1 probe ($\ell=1$) |
|---|---|---|
| 1…L | rising | high entropy / flat |
| L+1…2L | flat / saturated | **sharp rise** (outside resolves spurious latents) |


## 1. Setup & Imports

In [ ]:
import sys, os
import re
import math
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── workspace root ───────────────────────────────────────────────────────────
WORKSPACE = Path("/Users/parley/Documents/Last_token")
sys.path.insert(0, str(WORKSPACE))

import models   # models/transformer.py → models.CLM

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "mps"   # probing is lightweight; use "cuda" if available
print("PyTorch", torch.__version__, "| device:", DEVICE)


In [ ]:
# ── Grammar / dataset parameters ─────────────────────────────────────────────
# Adjust to match the training run you are probing.
V          = 8        # terminal vocabulary size
L          = 3         # grammar depth
BIN        = 2         # binary-rule multiplicity  (m_2 = BIN,  m_3 = BIN * V)
SEED_RULES = 12345678  # must match the training run

# ── Model hyper-parameters ────────────────────────────────────────────────────
# These cannot be inferred from the state-dict; set them to match training.
NUM_HEADS  = 16
# (embedding_dim, ffwd_size, depth are inferred automatically below)

# ── Probe dataset ─────────────────────────────────────────────────────────────
N_PROBE      = 32768   # number of fully-ternary sequences
PROBE_SEED   = 42

# ── Checkpoint ────────────────────────────────────────────────────────────────
SINGLE_CHECKPOINT = WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t512.pt"

print("Grammar  : v=%d, L=%d, bin=%d, seed_rules=%d" % (V, L, BIN, SEED_RULES))
print("Probe set: N=%d, seed=%d" % (N_PROBE, PROBE_SEED))
print("Checkpoint exists:", SINGLE_CHECKPOINT.exists())


## 2. Grammar Reconstruction & Probe Dataset

We reconstruct the grammar from the known seed, then generate **fully-ternary** sequences and track the ground-truth latent at every grammar level.

For a depth-$L$ all-ternary tree with $v^L = 27$ leaves:
- Token at input position $i$ has level-1 parent $\lfloor i/3 \rfloor$, level-2 parent $\lfloor i/9 \rfloor$, and root parent 0.


In [ ]:
_rhm_dir = str(WORKSPACE / "datasets" / "rhm")
if _rhm_dir not in sys.path:
    sys.path.insert(0, _rhm_dir)

from mixed_random_hierarchy_model import (
    sample_padded_rules,
    create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

m_2 = BIN          # binary-rule multiplicity
m_3 = BIN * V      # ternary-rule multiplicity
S_2, S_3 = 2, 3
D_MAX    = S_3 ** L        # max sequence length (27 for L=3, ternary)
T        = D_MAX - 1       # transformer block_size (26)
N_CLASSES = V              # num_classes == num_features

rules = sample_padded_rules(V, N_CLASSES, m_2, m_3, S_2, S_3, L, SEED_RULES)

# Ternary-only probabilities (put all weight on ternary rules)
ternary_probs = {}
for l in range(L):
    ternary_probs[l] = torch.cat([
        torch.zeros(m_2),
        torch.full((m_3,), 1.0 / m_3)
    ])

print(f"Grammar: v={V}, L={L}, m_2={m_2}, m_3={m_3}, d_max={D_MAX}, T={T}")
print(f"n_classes (root vocab) = {N_CLASSES}")


In [ ]:
def generate_ternary_with_latents(N, rules, V, L, m_2, m_3, ternary_probs, seed=42):
    """
    Generate N fully-ternary sequences and record the ground-truth latent at
    every grammar level.

    Returns
    -------
    terminals : LongTensor (N, D_MAX)   — terminal tokens (includes target at [:, -1])
    latents   : list of L tensors
        latents[0] : shape (N,)        — root label (level L)
        latents[1] : shape (N, 3)      — level L-1 latents
        ...
        latents[L-1] : shape (N, 3^(L-1))  — level-1 latents
    """
    torch.manual_seed(seed)
    n_classes = V   # num_classes == V

    root_labels = torch.randint(0, n_classes, (N,))
    features    = root_labels.clone()          # (N,)
    latents     = [root_labels.clone()]        # latents[0] = root

    for l in range(L - 1):
        # features: (N, 3^l) — expand one level downward
        flat = features.reshape(-1)            # (N * 3^l,)
        chosen = torch.multinomial(
            ternary_probs[l].unsqueeze(0).expand(flat.numel(), -1),
            num_samples=1, replacement=True,
        ).squeeze(-1).reshape(features.shape)  # (N, 3^l)
        # rules[l] has shape (V+1, m_2+m_3, S_3); each feature picks a rule
        features = rules[l][features, chosen].flatten(start_dim=1)  # (N, 3^(l+1))
        latents.append(features.clone())

    # Last expansion: level-1 → terminals
    flat = features.reshape(-1)
    chosen = torch.multinomial(
        ternary_probs[L - 1].unsqueeze(0).expand(flat.numel(), -1),
        num_samples=1, replacement=True,
    ).squeeze(-1).reshape(features.shape)
    terminals = rules[L - 1][features, chosen].flatten(start_dim=1)  # (N, D_MAX)

    return terminals, latents


terminals, latents = generate_ternary_with_latents(
    N_PROBE, rules, V, L, m_2, m_3, ternary_probs, seed=PROBE_SEED
)

print(f"terminals shape : {terminals.shape}")   # (N, 27)
for i, lat in enumerate(latents):
    print(f"latents[{i}] (level {L-i}) shape: {lat.shape}")


In [ ]:
# Build model inputs:  input = terminals[:, 0:T],  target = terminals[:, T]
# For fully-ternary sequences there is no padding (length = D_MAX exactly).
probe_inputs  = terminals[:, :T].long()   # (N, T=26)  — fed to the transformer
probe_targets = terminals[:, T].long()    # (N,)        — the prediction target

# ── Ground-truth label tensors for probing ─────────────────────────────────
# At input position j (0-indexed), the parse-tree ancestors are:
#   level-1 parent : latents[L-1][:, j // 3]
#   level-2 parent : latents[L-2][:, j // 9]
#   root  (level-L): latents[0]
#
# We flatten (N, T) → (N*T,) and index accordingly.

pos = torch.arange(T)                          # (T,)
# level-1: parent index in [0, 3^(L-1)-1]
level1_idx = pos // 3                          # (T,)
# level-2: parent index in [0, 3^(L-2)-1]
level2_idx = pos // 9                          # (T,)

# (N, T) label tensors
labels_root   = latents[0].unsqueeze(1).expand(N_PROBE, T)          # (N, T)
labels_level2 = latents[1][:, level2_idx]                           # (N, T)
labels_level1 = latents[L - 1][:, level1_idx]                       # (N, T)

# Flatten to (N*T,) — one data point per (sequence, position) pair
y_root   = labels_root.reshape(-1).numpy()    # root class label
y_level2 = labels_level2.reshape(-1).numpy()  # level-2 latent
y_level1 = labels_level1.reshape(-1).numpy()  # level-1 latent

print(f"probe_inputs  : {probe_inputs.shape}")
print(f"y_root   range: [{y_root.min()}, {y_root.max()}]  ({len(np.unique(y_root))} unique)")
print(f"y_level2 range: [{y_level2.min()}, {y_level2.max()}]")
print(f"y_level1 range: [{y_level1.min()}, {y_level1.max()}]")


## 3. Model Loading from Checkpoint

We infer most architecture parameters from the state-dict shapes. Only `num_heads` must be set manually (it does not appear in any tensor shape).


In [ ]:
def infer_arch(state_dict, num_heads):
    """Infer CLM constructor kwargs from a state_dict.  num_heads must be supplied."""
    vocab_size    = state_dict["token_embedding_table.weight"].shape[0]
    embedding_dim = state_dict["token_embedding_table.weight"].shape[1]
    block_size    = state_dict["blocks.0.attn.tril"].shape[0]
    num_layers    = max(
        int(k.split(".")[1]) for k in state_dict if k.startswith("blocks.")
    ) + 1
    ffwd_size     = (state_dict["blocks.0.ffwd.net.0.weight"].shape[0] // embedding_dim)
    rope          = "position_embedding_table.weight" not in state_dict
    share_emb     = (
        state_dict["token_embedding_table.weight"].data_ptr()
        == state_dict.get("lm_head.weight", torch.empty(0)).data_ptr()
    )
    return dict(
        vocab_size    = vocab_size,
        block_size    = block_size,
        embedding_dim = embedding_dim,
        num_heads     = num_heads,
        ffwd_size     = ffwd_size,
        num_layers    = num_layers,
        dropout       = 0.0,
        rope          = rope,
        share_emb     = False,
    )


def load_model_from_checkpoint(ckpt_path, num_heads, device="cpu"):
    """Load a CLM from a checkpoint file produced by main.py."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt["model"]
    arch = infer_arch(state_dict, num_heads)
    model = models.CLM(**arch)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    step  = ckpt.get("step", None)
    state = ckpt.get("state", {})
    return model, arch, step, state


# ── Use SINGLE_CHECKPOINT ─────────────────────────────────────────────────────
all_ckpts = [SINGLE_CHECKPOINT] if SINGLE_CHECKPOINT.exists() else []
print(f"Found {len(all_ckpts)} checkpoint files")
if all_ckpts:
    print("Checkpoint:", all_ckpts[0].name)


## 4. Activation Extraction via Forward Hooks

We register a hook on each `TransformerBlock` to capture the residual stream **after** that block (post-attention + post-FFN + skip connection). We also capture the embedding output (layer 0).

The result is a tensor of shape `(num_layers+1, N, T, embedding_dim)`.


In [ ]:
@torch.no_grad()
def extract_activations(model, inputs, device="cpu", batch_size=256):
    """
    Run `inputs` through `model` and return the residual stream at every layer.

    Parameters
    ----------
    model   : CLM (eval mode, frozen)
    inputs  : LongTensor (N, T)
    device  : str
    batch_size : int  — process in mini-batches to avoid OOM

    Returns
    -------
    activations : ndarray (num_probe_points, N, T, D)
        Index 0   = after token+position embedding (before any block)
        Index k>0 = after TransformerBlock k-1  (residual stream)
    """
    model.eval()
    num_layers = model.num_layers
    D          = model.embedding_dim
    N, Tseq    = inputs.shape
    num_points = num_layers + 1        # embedding + L blocks

    all_acts = [[] for _ in range(num_points)]   # list-of-lists

    for start in range(0, N, batch_size):
        batch = inputs[start:start + batch_size].to(device)
        B     = batch.shape[0]

        captured = {}   # hook_idx -> (B, T, D) tensor

        hooks = []

        # Hook to grab the embedding output (before any block)
        def _emb_hook(module, inp, out):
            captured[0] = out.detach().cpu()

        # We patch the forward pass manually to avoid modifying the model.
        # Strategy: temporarily replace model.blocks with a hooked Sequential.
        block_outputs = {}

        def make_block_hook(idx):
            def _hook(module, inp, out):
                block_outputs[idx] = out.detach().cpu()
            return _hook

        block_hooks = [
            block.register_forward_hook(make_block_hook(i))
            for i, block in enumerate(model.blocks)
        ]

        # Run forward pass; capture embedding separately via a hook on blocks[0]'s input
        # We intercept the embedding by hooking the ln_f or the first block input.
        # Simplest: hook the FIRST block's __call__ *input*.
        def _first_block_input_hook(module, inp, out):
            # inp[0] is the input to the first block = embedding output
            captured[0] = inp[0].detach().cpu()

        first_block_hook = model.blocks[0].register_forward_pre_hook(
            lambda m, inp: captured.update({0: inp[0].detach().cpu()})
        )

        _ = model(batch)   # forward pass triggers all hooks

        first_block_hook.remove()
        for h in block_hooks:
            h.remove()

        # Store results
        all_acts[0].append(captured[0])          # embedding
        for i in range(num_layers):
            all_acts[i + 1].append(block_outputs[i])  # after block i

    # Concatenate batches
    result = np.stack(
        [np.concatenate([a.numpy() for a in layer_acts], axis=0) for layer_acts in all_acts],
        axis=0,
    )  # (num_points, N, T, D)
    return result


# ── Sanity check on one checkpoint ───────────────────────────────────────────
if all_ckpts:
    _model, _arch, _step, _state = load_model_from_checkpoint(
        all_ckpts[-1], NUM_HEADS, device=DEVICE
    )
    print("Architecture:", {k: v for k, v in _arch.items() if k != "dropout"})
    print("Checkpoint step:", _step, " | val_loss:", _state.get("val_loss", "?"))

    _acts = extract_activations(_model, probe_inputs, device=DEVICE, batch_size=256)
    print("Activations shape (n_probe_points, N, T, D):", _acts.shape)
    del _model


## 5. Linear Probes

For each probe point (layer index) we train a `LogisticRegression` on the flattened `(N*T, D)` activations to predict each of the three latent levels.

We use a `StandardScaler → LogisticRegression(C=10, max_iter=1000)` pipeline, which is standard in probing literature. No regularisation tuning needed for this clean synthetic task.


In [ ]:
def train_probes(activations, y_root, y_level2, y_level1, test_frac=0.3, random_state=0):
    """
    Train a linear probe at each probe point (layer).

    Parameters
    ----------
    activations : ndarray (n_points, N, T, D)
    y_*         : ndarray (N*T,)  — ground-truth labels (flattened over sequences × positions)

    Returns
    -------
    results : dict with keys "root", "level2", "level1"
              each value is a list of length n_points containing probe accuracy (float)
    """
    n_points, N, Tseq, D = activations.shape
    NT = N * Tseq

    # Train / test split (positional — avoids data leakage between sequences)
    n_test  = int(NT * test_frac)
    n_train = NT - n_test
    idx     = np.random.RandomState(random_state).permutation(NT)
    train_idx, test_idx = idx[:n_train], idx[n_train:]

    results = {"root": [], "level2": [], "level1": []}
    label_map = {"root": y_root, "level2": y_level2, "level1": y_level1}

    for k in range(n_points):
        X = activations[k].reshape(NT, D)   # (NT, D)
        train_accs = {}
        for name, y in label_map.items():
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(C=10, max_iter=5000,
                                              solver="lbfgs", random_state=0,
                                              n_jobs=-1)),
            ])
            pipe.fit(X[train_idx], y[train_idx])
            train_accs[name] = pipe.score(X[train_idx], y[train_idx])
            acc = pipe.score(X[test_idx], y[test_idx])
            results[name].append(acc)

        print(f"  layer {k:2d} | "
              f"root={results['root'][-1]:.3f}(tr={train_accs['root']:.3f})  "
              f"lv2={results['level2'][-1]:.3f}(tr={train_accs['level2']:.3f})  "
              f"lv1={results['level1'][-1]:.3f}(tr={train_accs['level1']:.3f})")

    return results


# Chance-level baselines
chance_root   = 1.0 / N_CLASSES
chance_level2 = 1.0 / N_CLASSES   # level-2 latents are also drawn from V classes
chance_level1 = 1.0 / N_CLASSES
print(f"Chance level (all): {chance_root:.3f}  (1/{N_CLASSES})")


## 6. End-of-Training Probe: Single Checkpoint

Run the probing experiment on the **final** checkpoint to establish the *spatial* inside-outside structure.


In [ ]:
assert all_ckpts, "No checkpoint files found — check CHECKPOINT_DIR and CHECKPOINT_GLOB."

final_ckpt = all_ckpts[-1]
print(f"Probing checkpoint: {final_ckpt.name}")

model_final, arch_final, step_final, state_final = load_model_from_checkpoint(
    final_ckpt, NUM_HEADS, device=DEVICE
)
num_layers = arch_final["num_layers"]
print(f"num_layers={num_layers}, step={step_final}, val_loss={state_final.get('val_loss','?'):.4f}")

acts_final = extract_activations(model_final, probe_inputs, device=DEVICE, batch_size=256)
del model_final   # free memory

#print(f"\nTraining probes on activations {acts_final.shape} ...")
#probe_results_final = train_probes(acts_final, y_root, y_level2, y_level1)


## 7. Visualize End-of-Training Probe Results

Expected under the inside-outside hypothesis:
- **Root** ($\ell=3$): accuracy climbs mainly in layers 1–3 (inside pass builds the root representation).
- **Level-1** ($\ell=1$): accuracy climbs mainly in layers 4–6 (outside pass resolves local ambiguity).
- Both should be above chance well before the final layer.


In [ ]:
layer_labels = ["emb"] + [f"L{k+1}" for k in range(num_layers)]
x_ticks      = np.arange(len(layer_labels))

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(x_ticks, probe_results_final["root"],   "o-", color="tab:purple",
        linewidth=2.2, markersize=7, label=r"Root  $(\ell=L)$")
ax.plot(x_ticks, probe_results_final["level2"], "s-", color="tab:orange",
        linewidth=2.2, markersize=7, label=r"Level-2  $(\ell=L-1)$")
ax.plot(x_ticks, probe_results_final["level1"], "^-", color="tab:blue",
        linewidth=2.2, markersize=7, label=r"Level-1  $(\ell=1)$")

ax.axhline(chance_root, color="gray", linestyle=":", linewidth=1.5, label="Chance")
ax.axvline(num_layers // 2 + 0.5, color="black", linestyle="--", linewidth=1,
           label=f"Layer {num_layers//2 + 1} (inside/outside boundary)")

ax.set_xticks(x_ticks)
ax.set_xticklabels(layer_labels, fontsize=10)
ax.set_xlabel("Probe point (after embedding / transformer block)")
ax.set_ylabel("Linear probe accuracy")
ax.set_title(
    f"Linear probe accuracy vs layer  —  step {step_final}  "
    rf"($v={V}$, $L={L}$, bin={BIN})"
)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 7b. Last-Position Probe

Probe only the residual stream at position `T-1` (the last input token). In a causal LM this is the position that must aggregate the whole context to predict the next token, so the inside-outside structure should appear more sharply here than when averaging over all positions.

Labels at position `T-1 = 25`:
- **Root** — same as before (one root per sequence)
- **Level-2 parent** — `latents[1][:, 25 // 9]` = node index 2
- **Level-1 parent** — `latents[L-1][:, 25 // 3]` = node index 8

In [ ]:
# ── Last-position labels ──────────────────────────────────────────────────────
last_pos = T - 1   # = 25

y_root_last   = latents[0].numpy()                          # (N,)
y_level2_last = latents[1][:, last_pos // 9].numpy()        # (N,)
y_level1_last = latents[L - 1][:, last_pos // 3].numpy()   # (N,)

print(f"Last position: {last_pos}")
print(f"Level-2 node index: {last_pos // 9}  |  Level-1 node index: {last_pos // 3}")

PROBE_C = 0.1   # ← regularization strength; decrease if train >> test


def train_probes_last_pos(activations, y_root, y_level2, y_level1, test_frac=0.3, random_state=0):
    """Like train_probes but uses only the last token position."""
    n_points, N, Tseq, D = activations.shape
    X_all = activations[:, :, -1, :]   # (n_points, N, D)  — last position only

    idx       = np.random.RandomState(random_state).permutation(N)
    n_test    = int(N * test_frac)
    train_idx = idx[n_test:]
    test_idx  = idx[:n_test]

    results   = {"root": [], "level2": [], "level1": []}
    label_map = {"root": y_root, "level2": y_level2, "level1": y_level1}

    for k in range(n_points):
        X = X_all[k]   # (N, D)
        train_accs = {}
        for name, y in label_map.items():
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(C=PROBE_C, max_iter=5000,
                                              solver="lbfgs", random_state=0,
                                              n_jobs=-1)),
            ])
            pipe.fit(X[train_idx], y[train_idx])
            train_accs[name] = pipe.score(X[train_idx], y[train_idx])
            acc = pipe.score(X[test_idx], y[test_idx])
            results[name].append(acc)

        print(f"  layer {k:2d} | "
              f"root={results['root'][-1]:.3f}(tr={train_accs['root']:.3f})  "
              f"lv2={results['level2'][-1]:.3f}(tr={train_accs['level2']:.3f})  "
              f"lv1={results['level1'][-1]:.3f}(tr={train_accs['level1']:.3f})")

    return results


print(f"\nTraining last-position probes (C={PROBE_C}) on activations {acts_final.shape} ...")
probe_results_last = train_probes_last_pos(
    acts_final, y_root_last, y_level2_last, y_level1_last
)

In [ ]:
import pickle

with open("probe_results_step_512_32768_C_01.pkl", "wb") as f:
    pickle.dump(probe_results_last, f)

probe_results_step_512_32768_C_01=probe_results_last

In [ ]:
# ── Side-by-side: all positions vs last position ──────────────────────────────
fig, axes = plt.subplots(1, 1, figsize=(14, 5), sharey=True)
layer_labels = ["emb"] + [f"L{k+1}" for k in range(num_layers)]
x_ticks      = np.arange(len(layer_labels))

for ax, (results, title) in zip(axes, [
    (probe_results_last, "All positions (avg)"),
    (probe_results_last_16384_C_01,  f"Last position only (pos {last_pos})"),
]):
ax.plot(x_ticks, results["root"],   "o-", color="tab:purple",
        linewidth=2.2, markersize=7, label=r"Root  $(\ell=L)$")
ax.plot(x_ticks, results["level2"], "s-", color="tab:orange",
        linewidth=2.2, markersize=7, label=r"Level-2  $(\ell=L-1)$")
ax.plot(x_ticks, results["level1"], "^-", color="tab:blue",
        linewidth=2.2, markersize=7, label=r"Level-1  $(\ell=1)$")
ax.axhline(chance_root, color="gray", linestyle=":", linewidth=1.5, label="Chance")
ax.axvline(num_layers // 2 + 0.5, color="black", linestyle="--", linewidth=1,
            label=f"Layer {num_layers//2 + 1}")
ax.set_xticks(x_ticks)
ax.set_xticklabels(layer_labels, fontsize=10)
ax.set_xlabel("Probe point")
ax.set_title(title, fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)

axes[0].set_ylabel("Linear probe accuracy")
fig.suptitle(
    rf"Linear probe accuracy  —  step {step_final}  ($v={V}$, $L={L}$, bin={BIN})",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Single plot: all positions ───────────────────────────────────────────────
fig, ax= plt.subplots(1, 1, figsize=(7, 5))


import pickle

with open("probe_results_last_32768_C_01.pkl", "rb") as f:
    probe_results_last_32768_C_01 = pickle.load(f)
with open("probe_results_step_262144_32768_C_01.pkl", "rb") as f:
    probe_results_step_262144_32768_C_01 = pickle.load(f)
with open("probe_results_step_65536_32768_C_01.pkl", "rb") as f:
    probe_results_step_65536_32768_C_01 = pickle.load(f)
with open("probe_results_step_1024_32768_C_01.pkl", "rb") as f:
    probe_results_step_1024_32768_C_01 = pickle.load(f)
layer_labels = ["emb"] + [f"L{k+1}" for k in range(num_layers)]
x_ticks = np.arange(len(layer_labels))

results = probe_results_last_32768_C_01
results_2=probe_results_step_262144_32768_C_01
results_3=probe_results_step_65536_32768_C_01
results_4=probe_results_step_1024_32768_C_01
title = "Last position"

ax.plot(
    x_ticks, results["root"], "o-",
    color="tab:purple", linewidth=2.2, markersize=7,
    label=r"Root  $(\ell=L)$"
)
ax.plot(
    x_ticks, results_2["root"], "o--",
    color="tab:purple", linewidth=2.2, markersize=7,
    label=r"Root  $(\ell=L)$  (step 262144)"
)

ax.plot(
    x_ticks, results_3["root"], "o--",
    color="tab:purple", linewidth=2.2, markersize=7,
    label=r"Root  $(\ell=L)$  (step 65536)"
)
ax.plot(
    x_ticks, results_4["root"], "o--",
    color="tab:purple", linewidth=2.2, markersize=7,
    label=r"Root  $(\ell=L)$  (step 1024)"
)
ax.plot(
    x_ticks, results["level2"], "s-",
    color="tab:orange", linewidth=2.2, markersize=7,
    label=r"Level-2  $(\ell=L-1)$"
)

ax.plot(
    x_ticks, results_2["level2"], "s--",
    color="tab:orange", linewidth=2.2, markersize=7,
    label=r"Level-2  $(\ell=L-1)$  (step 262144)"
)

ax.plot(
    x_ticks, results_3["level2"], "s--",
    color="tab:orange", linewidth=2.2, markersize=7,
    label=r"Level-2  $(\ell=L-1)$  (step 65536)"
)

ax.plot(
    x_ticks, results_4["level2"], "s--",
    color="tab:orange", linewidth=2.2, markersize=7,
    label=r"Level-2  $(\ell=L-1)$  (step 1024)"
)
ax.plot(
    x_ticks, results["level1"], "^-",
    color="tab:blue", linewidth=2.2, markersize=7,
    label=r"Level-1  $(\ell=1)$"
)

ax.plot(
    x_ticks, results_2["level1"], "^--",
    color="tab:blue", linewidth=2.2, markersize=7,
    label=r"Level-1  $(\ell=1)$  (step 262144)"
)

ax.plot(
    x_ticks, results_3["level1"], "^--",
    color="tab:blue", linewidth=2.2, markersize=7,
    label=r"Level-1  $(\ell=1)$  (step 65536)"
)

ax.plot(
    x_ticks, results_4["level1"], "^--",
    color="tab:blue", linewidth=2.2, markersize=7,
    label=r"Level-1  $(\ell=1)$  (step 1024)"
)

ax.axhline(
    chance_root,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label="Chance"
)

# ax.axvline(
#     num_layers // 2 + 0.5,
#     color="black",
#     linestyle="--",
#     linewidth=1,
#     label=f"Layer {num_layers // 2 + 1}"
# )

ax.set_xticks(x_ticks)
ax.set_xticklabels(layer_labels, fontsize=10)
ax.set_xlabel("Probe point")
ax.set_ylabel("Linear probe accuracy")
ax.set_title(title, fontsize=12)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
#ax.legend(fontsize=10)

fig.suptitle(
    rf"Linear probe accuracy  —  step {step_final}  ($v={V}$, $L={L}$, bin={BIN})",
    fontsize=13,
)

plt.tight_layout()
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Load probe results
# ---------------------------------------------------------------------
with open("probe_results_step_512_32768_C_01.pkl", "rb") as f:
    probe_results_step_512_32768_C_01 = pickle.load(f)

with open("probe_results_step_65536_32768_C_01.pkl", "rb") as f:
    probe_results_step_65536_32768_C_01 = pickle.load(f)

with open("probe_results_step_262144_32768_C_01.pkl", "rb") as f:
    probe_results_step_262144_32768_C_01 = pickle.load(f)

with open("probe_results_last_32768_C_01.pkl", "rb") as f:
    probe_results_last_32768_C_01 = pickle.load(f)


# ---------------------------------------------------------------------
# Labels and x-axis
# ---------------------------------------------------------------------
num_layers=6
layer_labels = ["emb"] + [f"L{k+1}" for k in range(num_layers)]
x_ticks = np.arange(len(layer_labels))


# ---------------------------------------------------------------------
# Learning stages
# Ordered from earliest to latest
# ---------------------------------------------------------------------
stage_results = [
    #probe_results_step_512_32768_C_01,
    probe_results_step_65536_32768_C_01,
    probe_results_step_262144_32768_C_01,
    probe_results_last_32768_C_01,
]

stage_labels = [
    #r"$\mathcal{L}_{\ell=1}^{\mathrm{amb}}$",
    r"$\mathcal{L}_{\ell=1}$",
    r"$\mathcal{L}_{\ell=2}$",
    r"$\mathcal{L}_{\ell=3}$",
]

stage_linestyles = [
    ":",
    "--",
    "-.",
    "-",
]

stage_markers = [
    "o",
    "s",
    "^",
    "D",
]


# ---------------------------------------------------------------------
# Probed objects: one per subplot
# ---------------------------------------------------------------------
probe_keys = [
    "level1",
    "level2",
    "root",
]

subplot_titles = [
    "First level latent",
    "Second level latent",
    "Root",
]


# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)

for ax, key, title in zip(axes, probe_keys, subplot_titles):

    for results, label, linestyle, marker in zip(
        stage_results,
        stage_labels,
        stage_linestyles,
        stage_markers,
    ):
        ax.plot(
            x_ticks,
            results[key],
            marker=marker,
            linestyle=linestyle,
            linewidth=2.2,
            markersize=6.5,
            label=label,
        )

    ax.axhline(
        chance_root,
        color="gray",
        linestyle=":",
        linewidth=1.5
    )

    ax.set_xticks(x_ticks)
    ax.set_xticklabels(layer_labels, fontsize=10)
    ax.set_xlabel("Transformer layer")
    ax.set_title(title, fontsize=13)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Linear probe accuracy")

# One shared legend for the whole figure
# One shared legend for the whole figure
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    fontsize=11,
    title="Model performance (on ternary tree) at probed checkpoint",
    title_fontsize=12,
    frameon=True,
    bbox_to_anchor=(0.5, -0.08),
)

fig.suptitle(
    rf"Linear probe accuracy at last position  ($v={V}$, $L={L}$, $f=1/4$)",
    fontsize=14,
)

plt.tight_layout(rect=[0, 0.12, 1, 0.93])
plt.show()

## 8. Cross-Training Probing: Accuracy vs Layer vs Training Step

Iterate over **all** checkpoints, extract activations, and train probes. This gives a 2D matrix `(n_steps, n_probe_points)` per latent level, revealing *when* the inside and outside passes emerge during training.

> This cell is slow — it runs one full probe experiment per checkpoint. Set `CKPT_STRIDE` to subsample if needed.


In [ ]:
CKPT_STRIDE = 1   # set to >1 to probe every k-th checkpoint

selected_ckpts = all_ckpts[::CKPT_STRIDE]
print(f"Probing {len(selected_ckpts)} checkpoints ...")

timeline = []   # list of dicts: {step, val_loss, results}

for ckpt_path in selected_ckpts:
    model_ck, _, step_ck, state_ck = load_model_from_checkpoint(
        ckpt_path, NUM_HEADS, device=DEVICE
    )
    acts_ck = extract_activations(model_ck, probe_inputs, device=DEVICE, batch_size=256)
    del model_ck

    print(f"\n--- step {step_ck}  val_loss={state_ck.get('val_loss','?'):.4f} ---")
    results_ck = train_probes(acts_ck, y_root, y_level2, y_level1)
    timeline.append({
        "step":     step_ck,
        "val_loss": state_ck.get("val_loss", float("nan")),
        "results":  results_ck,
    })

steps_arr   = np.array([t["step"]     for t in timeline])
val_losses  = np.array([t["val_loss"] for t in timeline])
n_steps     = len(timeline)
n_points    = len(timeline[0]["results"]["root"])

# Build 2-D accuracy matrices  (n_steps, n_points)
mat_root   = np.array([t["results"]["root"]   for t in timeline])
mat_level2 = np.array([t["results"]["level2"] for t in timeline])
mat_level1 = np.array([t["results"]["level1"] for t in timeline])

print("\nShapes:", mat_root.shape)


## 9. Visualize Cross-Training Results

Three panels: heatmaps of probe accuracy over (training step × layer) for each latent level. White = chance; dark = perfect decoding.

Below the heatmaps: accuracy curves per layer vs training step, with the validation-loss milestones ($\mathcal{L}_{\ell=1}$, $\mathcal{L}_{\ell=2}$, $\mathcal{L}_{\ell=3}$) overlaid on a secondary axis.


In [ ]:
# ── Theoretical loss thresholds (adjust v/f/s to your grammar) ───────────────
_f = BIN / V       # branching fraction
_v, _s = V, L
_m  = _f * _v**2
_ell1 = 1 + (_v - 1) * (_m * _v - 1) / (_v**_s - 1)
log_1 = np.log(_ell1)
_ell2 = 1 + ((_v - 1) / (_v**_s - 1)) * (_m * (1 + (_v - 1) * (_m * _v - 1) / (_v**_s - 1)) - 1)
log_2 = np.log(_ell2)
_ell3 = 1 + ((_v - 1) / (_v**_s - 1)) * (_m * _ell2 - 1)
log_3 = np.log(_ell3)
print(f"log_1={log_1:.4f}  log_2={log_2:.4f}  log_3={log_3:.4f}")

# ── Panel 1: heatmaps ─────────────────────────────────────────────────────────
matrices = {
    rf"Root  $(\ell={L})$":   mat_root,
    rf"Level-2  $(\ell={L-1})$": mat_level2,
    rf"Level-1  $(\ell=1)$":  mat_level1,
}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (title, mat) in zip(axes, matrices.items()):
    im = ax.imshow(
        mat.T,                       # rows = layers, cols = time
        aspect="auto",
        origin="lower",
        vmin=chance_root, vmax=1.0,
        cmap="viridis",
        extent=[0, n_steps, -0.5, n_points - 0.5],
    )
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Checkpoint index (earlier → later)", fontsize=10)
    if ax is axes[0]:
        ax.set_ylabel("Probe point")
    ax.set_yticks(range(n_points))
    ax.set_yticklabels(layer_labels, fontsize=9)
    ax.axhline(num_layers // 2 + 0.5, color="white", linestyle="--", linewidth=1.2)
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)

fig.suptitle(
    rf"Probe accuracy heatmap (step × layer)  —  $v={V}$, $L={L}$, bin={BIN}",
    fontsize=13,
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Panel 2: accuracy per layer vs training step, one subplot per latent level ─
colors_by_layer = plt.get_cmap("plasma")(np.linspace(0.1, 0.9, n_points))

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=False)

plot_data = [
    (mat_root,   rf"Root  $(\ell={L})$",      "tab:purple"),
    (mat_level2, rf"Level-2  $(\ell={L-1})$", "tab:orange"),
    (mat_level1, rf"Level-1  $(\ell=1)$",     "tab:blue"),
]

for ax, (mat, title, _) in zip(axes, plot_data):
    for k in range(n_points):
        ax.plot(
            steps_arr, mat[:, k],
            color=colors_by_layer[k],
            linewidth=1.8,
            label=layer_labels[k],
        )
    ax.axhline(chance_root, color="gray", linestyle=":", linewidth=1.2, label="Chance")
    ax.set_title(title, fontsize=12)
    ax.set_xscale("log")
    ax.set_xlabel("Training step", fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

    # secondary y-axis: validation loss
    ax2 = ax.twinx()
    ax2.plot(steps_arr, val_losses, color="black", linestyle="--",
             linewidth=1.5, alpha=0.5, label="val loss")
    ax2.axhline(log_1, color="red",    linestyle="--", linewidth=1, alpha=0.7)
    ax2.axhline(log_2, color="green",  linestyle="--", linewidth=1, alpha=0.7)
    ax2.axhline(log_3, color="purple", linestyle="--", linewidth=1, alpha=0.7)
    ax2.set_ylabel("Val loss", fontsize=9)
    ax2.invert_yaxis()

axes[0].set_ylabel("Linear probe accuracy", fontsize=11)
axes[0].legend(title="Layer", ncol=2, fontsize=8, loc="lower right")

fig.suptitle(
    rf"Probe accuracy vs training step per layer  —  $v={V}$, $L={L}$, bin={BIN}",
    fontsize=13,
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Save probe results for later analysis ─────────────────────────────────────
np.savez(
    WORKSPACE / "probe_results.npz",
    steps        = steps_arr,
    val_losses   = val_losses,
    mat_root     = mat_root,
    mat_level2   = mat_level2,
    mat_level1   = mat_level1,
    layer_labels = np.array(layer_labels),
    num_layers   = num_layers,
    V=V, L=L, BIN=BIN,
)
print("Saved probe_results.npz")
